# Validating Assignments (`validate_assignment`)

Pydantic models are inherently **mutable** objects (unless explicitly frozen). Once a model is instantiated, you can change the values of its fields using standard Python dot notation.

However, by default, Pydantic **does not validate data on reassignment**. It only validates data during the initial loading/deserialization phase.

## 1. The Assignment Loophole

If you do not explicitly instruct Pydantic to validate assignments, you can accidentally inject bad state into a previously validated object.

```python
from pydantic import BaseModel

class User(BaseModel):
    age: int

# Phase 1: Creation. Pydantic validates this successfully.
u1 = User(age=25)

# Phase 2: Mutation. Pydantic IGNORES validation here.
u1.age = "twenty-five"

print(u1.age) 
# Output: "twenty-five" (Your model now contains a string where an int is required!)

```

## 2. Closing the Loophole (`validate_assignment=True`)

To enforce strict type contracts throughout the entire lifecycle of an object, you must enable `validate_assignment=True` in the model configuration.

```python
from pydantic import ConfigDict, ValidationError

class StrictUser(BaseModel):
    model_config = ConfigDict(validate_assignment=True)
    age: int

u2 = StrictUser(age=25)

try:
    u2.age = "twenty-five"
except ValidationError as e:
    print(e)
    # Output: 1 validation error for StrictUser
    # age: Input should be a valid integer

```

### Why is this False by default?

Like `validate_default`, this boils down to **performance**.
If you have a complex Pydantic model and you run a `for` loop that updates its fields 1,000 times, you do not want Pydantic to execute its heavy validation engine 1,000 times on trusted internal code. Pydantic assumes that once the raw data crosses the boundary (e.g., from an API payload into Python) and passes initial validation, the Python developer will write correct code to update it thereafter.

You should only enable `validate_assignment=True` in highly strict architectures where objects are frequently updated dynamically from untrusted sources.


### Interview Preparation: Validating Assignments

<details>
<summary><b>1. A junior developer complains that Pydantic is broken because they created a `User` model with `age: int`, but they were able to run `user.age = "String"` without getting an error. Explain why Pydantic behaves this way.</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Pydantic is a parsing and validation library primarily designed to validate data *at the boundaries* (e.g., when a JSON payload is deserialized into a Python object). <br><br>
By default, it does not validate subsequent assignments (mutations) on the object. This is a deliberate performance optimization. Pydantic assumes that once the data is safely loaded into the application, the developer will manage state correctly. Running the validation engine on every single object mutation would drastically slow down execution speed in complex applications. 
</details>

<details>
<summary><b>2. How do you force Pydantic to run validation checks every time an attribute is modified?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You add `model_config = ConfigDict(validate_assignment=True)` to the class definition. This instructs Pydantic to intercept any attribute assignment via dot notation and run it through the validation and coercion pipeline before updating the state.
</details>

<details>
<summary><b>3. You have a model configured with `validate_assignment=True`. The field is defined as `price: float`. You execute `model.price = 15` (an integer). Does it raise an error?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
No, it does not raise an error. <br><br>
Because Pydantic operates in Lax Coercion mode by default, the assignment validation intercepts the integer `15`, realizes it can be safely coerced into a float, and stores it as `15.0`. It would only raise an error if you assigned a type that could not be coerced (like a non-numeric string), or if you had also configured the model with `strict=True`.
</details>

<details>
<summary><b>4. You are designing a data pipeline. When would you choose `frozen=True` versus `validate_assignment=True`? Can you use both?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
You use `frozen=True` when you want the object to be completely immutable (and therefore hashable, e.g., for use as a dictionary key or in strict functional programming). <br><br>
You use `validate_assignment=True` when your application *requires* mutating the object's state dynamically, but you want to guarantee that those mutations never violate the schema's type constraints. <br><br>
You cannot logically use both. If a model is frozen, assignments are impossible, meaning `validate_assignment=True` becomes dead code.
</details>